# 00 – Download Remote Sensing Data
**Purpose:** Download SWOT satellite observations for SWORD reaches in a defined AOI.

**Workflow:**
0 | Imports
- 0.1 | Configuration
- 0.2 | Hydrocron API (no login required)
- 0.3 | earthaccess (NASA login required)

**References:**
- SWOT Mission: https://swot.jpl.nasa.gov
- Hydrocron API: https://github.com/podaac/hydrocron
- PO.DAAC SWOT Data: (https://podaac.jpl.nasa.gov/SWOT)
- Tebaldi, N., Nickels, C. (2025): [*Hydrocron API: SWOT Time Series Examples.*](https://podaac.github.io/tutorials/notebooks/datasets/Hydrocron_SWOT_timeseries_examples.html)
- Nickels, C. (2025): [*Search and Download SWOT Data via <code>earthaccess</code>*. ](https://podaac.github.io/tutorials/notebooks/SearchDownload_SWOTviaCMR.html)
- earthaccess docs: (https://earthaccess.readthedocs.io)
- NASA Earthdata Login: (https://urs.earthdata.nasa.gov)

To-Do:
- [] Thinking about variables which can be calculated by SWOT data and which are not in SWORD dataset
- [] making the workflow for investigation and download and variable calculation before download smarter.
- [] do I even need to download? I could just ask API for values, calculate new ones and join them to the SWORD dataset?
- [] How should can I besser handle the amount of AWOT data? Maybe...by..maybe creating a Goal-Reach_list and using parquet instead of shp and gpkg.

---
---
## 0 | Imports


In [ ]:
import geopandas as gpd
import os
import sys
import earthaccess

from soilgrids import soilgrids  

sys.path.append(os.path.join("..", "src"))
from _0_config_paths import DATA_RAW, DATA_PROCESSED, DATA_OUTPUT
from _00_download_rs_data import (
    query_hydrocron_bulk,
    parse_hydrocron_results,
    compute_bbox,
    search_swot_granules,
    stream_swot_granules,
    download_soilgrids,
    download_worldcover,
    merge_raster_tiles
)

---
---
## 0.1 | Configuration

In [ ]:

STUDY_AREA = "naryn"

# CONFIGURATION
# INPUT: clipped SWORD from Notebook 01 
IN_SWORD = os.path.join(DATA_PROCESSED, "sword_naryn_clip.gpkg")

#--- TIME RANGE for SWOT observations -------------------------------------
# Format: ISO 8601 (YYYY-MM-DDTHH:MM:SSZ)
START_TIME = "2025-08-01T00:00:00Z" # Earliest date: 2023-08-01T00:00:00Z
END_TIME = "2025-08-31T00:00:00Z"


#--- HYDROCRON: which SWOT fields to retrieve per reach --------------------
# NOTE: For full list of fields look in "_0_download_rs.py" file
HYDROCRON_FIELDS = "reach_id,time_str,wse ,slope ,width ,p_dist_out ,geometry"
HYDROCRON_OUTPUT_FORMAT = "geojson" # "geojson" or "csv"
HYDROCRON_REQUEST_DELAY = 0.3 # delay between API calls
MIN_WIDTH_M             = 100  

#--- EARTHACCESS: which product collection to download ---------------------
# Most relevant collections:
#   SWOT_L2_HR_RiverSP_reach_D: River reaches (Shapefile, ~10 km segments)
#   SWOT_L2_HR_RiverSP_node_D: River nodes (Shapefile, ~200 m spacing)
#   SWOT_L2_HR_Raster_D: Raster WSE/water mask (netCDF, large files)

# NOTE: "_D" suffix = Version D (latest as of 2025). Change if newer version exists.
EARTHACCESS_COLLECTION = "SWOT_L2_HR_RiverSP_reach_D"
GRANULE_STRIDE = 1     # 1 = every pass, 3 = every third pass
MAX_GRANULES = None  # None = all granules, integer = cap for testing

OUT_WORLDCOVER_DIR = os.path.join(DATA_RAW, "0_data", "worldcover")

#--- OUTPUT paths ---------------------------------------------------------
OUT_HYDROCRON  = os.path.join(DATA_PROCESSED, "swot_hydrocron.gpkg")
OUT_EARTHACCESS = os.path.join(DATA_PROCESSED, "swot_earthaccess.gpkg")
OUT_SOILGRIDS_DIR = os.path.join(DATA_RAW, "0_data", "soilgrids")

# ESA worldcover
OUT_WORLDCOVER_DIR = os.path.join(DATA_RAW, "0_data", "worldcover")
OUT_WORLDCOVER_MERGED = os.path.join(
    DATA_RAW, "0_data", "worldcover", "worldcover_merged.tif"
)

# soilgrids
# Depth intervals available: 0-5cm, 5-15cm, 15-30cm, 30-60cm
# Full variable list: https://www.isric.org/explore/soilgrids
OUT_SOILGRIDS_DIR = os.path.join(DATA_RAW, "0_data", "soilgrids")

SOILGRIDS_VARS = [
    # (service_id, coverage_id, description)

    # Soil Texture – controls sediment transport and bank stability
    ("clay", "clay_0-5cm_mean", "Clay content (%) – cohesive banks, meandering tendency"),
    ("sand", "sand_0-5cm_mean", "Sand content (%) – non-cohesive banks, braided tendency"),
    ("silt", "silt_0-5cm_mean", "Silt content (%) – intermediate grain size"),

    # Soil Structure
    ("cfvo", "cfvo_0-5cm_mean", "Coarse fragments volume (%) – rocky soils, mountain reaches"),
]

In [4]:
# Load clipped SWORD
sword = gpd.read_file(IN_SWORD)
# Compute AOI bounding box from SWORD extent
bbox = compute_bbox(sword)


Auto-computed bbox (with 0.05 buffer):
  (lon_min=71.7047, lat_min=40.8544, lon_max=78.2982, lat_max=42.3658)


---
---
## 0.2 | Hydrocron API

Queries SWOT time series per SWORD reach. No file download, no NASA login required.
SWOT observes rivers wider than ca. 100 m, narrower reaches are skipped automatically.

Needing the **Hydrocron API** (PO.DAAC / NASA JPL) for each SWORD reach in the AOI.
Returns all SWOT observations within the configured time range as GeoJSON or CSV.

**How:** SWOT data is archived as globally-tiled shapefiles per satellite pass
(one file = one pass over a whole continent). Hydrocron unpacks these shapefiles and
re-indexes them by reach_id, so a single API call returns all observations for one reach
across all passes --> without downloading any file. **No NASA login required.**

https://github.com/podaac/hydrocron

In [ ]:
print(f"SWORD reaches loaded: {len(sword)} | CRS: {sword.crs}")

# Query all reaches via Hydrocron API
all_features = query_hydrocron_bulk(
    sword = sword,
    start_time= START_TIME,
    end_time = END_TIME,
    fields = HYDROCRON_FIELDS,
    min_width_m = MIN_WIDTH_M,
    request_delay = HYDROCRON_REQUEST_DELAY
)

In [ ]:
# Parse raw GeoJSON into clean GeoDataFrame and save
swot_gdf = parse_hydrocron_results(all_features)

if swot_gdf is not None:
    num_cols = [c for c in ["wse", "slope", "width"] if c in swot_gdf.columns]
    print(swot_gdf[num_cols].describe().round(3))
    print(f"\nObservations per reach:")
    print(swot_gdf.groupby("reach_id").size().describe())

    os.makedirs(DATA_PROCESSED, exist_ok=True)
    swot_gdf.to_file(OUT_HYDROCRON, driver="GPKG")
    print(f"\nSaved: {OUT_HYDROCRON}")

---
---
## 0.3 | earthaccess – Download SWOT granules

Downloads the actual SWOT Shapefile granules (one file = one satellite pass over a
continent). Files are filtered spatially by bounding box and temporally by date range.

**downloadable in this section:**
- Raster products (`L2_HR_Raster`)
- Full Shapefile attributes (not just the Hydrocron field subset)

**!!!Warning:!!!** Each SWOT granule covers an entire continent pass and can be **several
hundred MB to several GB**. After download the data is clipped to the AOI.

**NASA Earthdata Login required.** acc at: https://urs.earthdata.nasa.gov

References:
- earthaccess: https://earthaccess.readthedocs.io
- SWOT collections: https://podaac.github.io/tutorials/quarto_text/SWOT.html
- PO.DAAC Cookbook: https://podaac.github.io/tutorials

In [ ]:
# EARTHACCESS: authenticate with NASA Earthdata Login

# earthaccess.login() will:
#   Try to use a stored .netrc file silently
#   If not found: prompt for username + password interactively

auth = earthaccess.login(strategy="interactive", persist=True)
print(f"Login successful: {auth.authenticated}")

In [ ]:
# Search for matching granules
results = search_swot_granules(
    collection = EARTHACCESS_COLLECTION,
    bbox = bbox,
    start_time = START_TIME,
    end_time = END_TIME
)

# Stream granules in-memory, clip to AOI, merge into GeoDataFrame
swot_ea = stream_swot_granules(
    results = results,
    aoi_bounds = sword.total_bounds,
    max_granules = MAX_GRANULES,
    granule_stride = GRANULE_STRIDE
)

if swot_ea is not None:
    os.makedirs(DATA_PROCESSED, exist_ok=True)
    swot_ea.to_file(OUT_EARTHACCESS, driver="GPKG")
    print(f"Saved: {OUT_EARTHACCESS}")

---
## 0.4 | Download ESA WorldCover

In [5]:
# sword and bbox were already loaded in Section 0.2
worldcover_files = download_worldcover(
    bbox    = bbox,
    out_dir = OUT_WORLDCOVER_DIR
)
print(f"WorldCover tiles: {worldcover_files}")

Tiles needed: 8
  lon=69, lat=39
  lon=69, lat=42
  lon=72, lat=39
  lon=72, lat=42
  lon=75, lat=39
  lon=75, lat=42
  lon=78, lat=39
  lon=78, lat=42
Downloading: ESA_WorldCover_10m_2021_v200_N39E069_Map.tif
  Saved: D:\0_InnoLab\0_data\WorldCover\ESA_WorldCover_10m_2021_v200_N39E069_Map.tif
Downloading: ESA_WorldCover_10m_2021_v200_N42E069_Map.tif
  Saved: D:\0_InnoLab\0_data\WorldCover\ESA_WorldCover_10m_2021_v200_N42E069_Map.tif
Downloading: ESA_WorldCover_10m_2021_v200_N39E072_Map.tif
  Saved: D:\0_InnoLab\0_data\WorldCover\ESA_WorldCover_10m_2021_v200_N39E072_Map.tif
Downloading: ESA_WorldCover_10m_2021_v200_N42E072_Map.tif
  Saved: D:\0_InnoLab\0_data\WorldCover\ESA_WorldCover_10m_2021_v200_N42E072_Map.tif
Downloading: ESA_WorldCover_10m_2021_v200_N39E075_Map.tif
  Saved: D:\0_InnoLab\0_data\WorldCover\ESA_WorldCover_10m_2021_v200_N39E075_Map.tif
Downloading: ESA_WorldCover_10m_2021_v200_N42E075_Map.tif
  Saved: D:\0_InnoLab\0_data\WorldCover\ESA_WorldCover_10m_2021_v200_N42E07

In [6]:
worldcover_merged = merge_raster_tiles(
    tile_paths = worldcover_files,
    out_path   = OUT_WORLDCOVER_MERGED
)

Merging 8 tiles...
Merged 8 tiles → D:\0_InnoLab\0_data\WorldCover\WorldCover_merged.tif


---
## 0.5 | Download soilgrids


In [ ]:
soilgrids_files = download_soilgrids(
    bbox           = bbox,
    out_dir        = OUT_SOILGRIDS_DIR,
    soilgrids_vars = SOILGRIDS_VARS
)
print(f"soilgrids files: {soilgrids_files}")

Requesting grid: 2927 x 671 pixels
Downloading: clay_0-5cm_mean – Clay content (%) – cohesive banks, meandering tendency
  Saved: D:\0_InnoLab\0_data\SoilGrids\clay_0-5cm_mean.tif
Downloading: sand_0-5cm_mean – Sand content (%) – non-cohesive banks, braided tendency
  Saved: D:\0_InnoLab\0_data\SoilGrids\sand_0-5cm_mean.tif
Downloading: silt_0-5cm_mean – Silt content (%) – intermediate grain size
  Saved: D:\0_InnoLab\0_data\SoilGrids\silt_0-5cm_mean.tif
Downloading: cfvo_0-5cm_mean – Coarse fragments volume (%) – rocky soils, mountain reaches
  Saved: D:\0_InnoLab\0_data\SoilGrids\cfvo_0-5cm_mean.tif

Download complete: 4 variables
SoilGrids files: ['D:\\0_InnoLab\\0_data\\SoilGrids\\clay_0-5cm_mean.tif', 'D:\\0_InnoLab\\0_data\\SoilGrids\\sand_0-5cm_mean.tif', 'D:\\0_InnoLab\\0_data\\SoilGrids\\silt_0-5cm_mean.tif', 'D:\\0_InnoLab\\0_data\\SoilGrids\\cfvo_0-5cm_mean.tif']
